# Methods paper — Diagnostics Notebook 01

**Purpose.** Quantify *why* the original multimodal-BD pipeline reported 82.6% macro-F1 while passing the standard
validation battery. Every experiment here is a diagnostic; nothing is tuned to improve performance.

**Part A** runs on the exact arrays behind the reported numbers (`multi_source_results/validation_data.npz`), if present.
**Part B** rebuilds the pipeline from the raw datasets, *with provenance metadata* (subject IDs, window indices, source corpus),
so leakage and circularity can be measured.

**How to run.** Runtime → Change runtime type → GPU (only the BERT step needs it). Then Runtime → Run all.
If a cell stops with a `CONFIG NEEDED` message, fix the setting it names in cell 0 and re-run from there.

**What to send back.** The block printed by the final cell (between the `=== PASTE BACK ===` markers),
plus `results.json` from `OUT_DIR` if anything looks off.

Expected runtime: ~10 min for Part A, 30–60 min for Part B (BERT embeddings are cached after the first run).

In [9]:
#@title 0 · Setup & configuration
import os, json, pickle, time, warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

SMOKE_TEST = os.environ.get('BD_SMOKE_TEST') == '1'   # internal: synthetic-data test of this notebook

if not SMOKE_TEST:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('Not running in Colab: set the paths below to your local copies.')

# ---- Paths: copied from the original notebooks. Note the trailing spaces in some folder names. ----
DEV = '/content/drive/MyDrive/Research on BD NYU /development'
PATHS = {
    'WESAD':          f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD',
    'OBF':            f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/OBF-Psychiatric Dataset',
    'KAGGLE_BD_CSV':  f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/Kaggle_BD /bipolar_dataset.csv',
    'MULTICLASS_CSV': f'/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/multi class depression dataset/dataset.csv',
    # Saved by the original run; read by control_experimentation_multi_fusion.ipynb
    'ORIGINAL_RESULTS_DIR': '/content/drive/MyDrive/Research on BD NYU/multi_source_results',
}
# Written to your own Drive so it works even if the shared folder is read-only for you.
OUT_DIR = '/content/drive/MyDrive/bd_methods_paper/run_01'

# ---- OBF column names (auto-detected in Part B; set explicitly if detection fails) ----
OBF_ID_COL = None      # column identifying the person (e.g. 'number')
OBF_DIAG_COL = None    # column with the clinical diagnosis / group

# ---- Experiment sizes ----
SEED = 42
N_SHUFFLES = 20        # shuffle trials per condition (Part A)
N_CV_REPEATS = 5       # repeats of 5-fold CV
N_BUILDS = 10          # independent re-draws of the label-conditional pairing (Part B waterfall)
WATERFALL_CV_REPEATS = 3
BERT_BATCH = 64

if SMOKE_TEST:
    base = os.environ['BD_SMOKE_DIR']
    PATHS = {'WESAD': f'{base}/WESAD', 'OBF': f'{base}/OBF', 'KAGGLE_BD_CSV': f'{base}/kaggle.csv',
             'MULTICLASS_CSV': f'{base}/multiclass.csv', 'ORIGINAL_RESULTS_DIR': f'{base}/multi_source_results'}
    OUT_DIR = f'{base}/out'
    N_SHUFFLES, N_CV_REPEATS, N_BUILDS, WATERFALL_CV_REPEATS = 3, 2, 2, 1

CACHE_DIR = f'{OUT_DIR}/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold, StratifiedGroupKFold,
                                     GroupKFold, StratifiedShuffleSplit)
from sklearn.metrics import f1_score, adjusted_rand_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

CLASS_NAMES = ['Euthymia', 'Depression', 'Mania']
RESULTS = {'meta': {'started': time.strftime('%Y-%m-%d %H:%M'), 'smoke_test': SMOKE_TEST}}

def save_results():
    def conv(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return str(o)
    with open(f'{OUT_DIR}/results.json', 'w') as f:
        json.dump(RESULTS, f, indent=2, default=conv)

def rf(seed=SEED):
    """Identical hyperparameters to the original pipeline."""
    return RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=seed, n_jobs=-1)

def macro_f1(y, p):
    return float(f1_score(y, p, average='macro'))

def original_split(y):
    """The exact split used for every reported number: stratified 80/20, random_state=42."""
    return train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)

def split_f1(X, y, model_seed=SEED):
    tr, te = original_split(y)
    return macro_f1(y[te], rf(model_seed).fit(X[tr], y[tr]).predict(X[te]))

def _folds(y, groups, seed):
    if groups is None:
        return StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    k = min(5, len(np.unique(groups)))
    return StratifiedGroupKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, groups)

def cv_f1(X, y, groups=None, n_repeats=N_CV_REPEATS, seed=SEED):
    """Repeated 5-fold CV. Out-of-fold predictions are pooled per repeat; returns (mean, sd) over repeats."""
    scores = []
    for r in range(n_repeats):
        oof = np.empty_like(y)
        for tr, te in _folds(y, groups, seed + r):
            oof[te] = rf(seed + r).fit(X[tr], y[tr]).predict(X[te])
        scores.append(macro_f1(y, oof))
    return float(np.mean(scores)), float(np.std(scores))

def perm_index(n, rng, y=None):
    """Row permutation. With y: permute only within each class (label-preserving shuffle)."""
    if y is None:
        return rng.permutation(n)
    idx = np.arange(n)
    for c in np.unique(y):
        pos = np.where(y == c)[0]
        idx[pos] = rng.permutation(pos)
    return idx

def ece(y_true, y_prob, n_bins=10):
    """Confidence ECE, identical to compute_calibration_metrics() in the original control notebook."""
    conf = y_prob.max(1); acc = (y_prob.argmax(1) == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += abs(conf[m].mean() - acc[m].mean()) * m.mean()
    return float(e)

def fmt(m, s=None):
    return f'{m:.3f}' if s is None else f'{m:.3f} ± {s:.3f}'

print('OUT_DIR:', OUT_DIR)
for k, p in PATHS.items():
    print(f'  {k:22s} exists={os.path.exists(p)}  {p}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT_DIR: /content/drive/MyDrive/bd_methods_paper/run_01
  WESAD                  exists=True  /content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD
  OBF                    exists=True  /content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/OBF-Psychiatric Dataset
  KAGGLE_BD_CSV          exists=True  /content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/Kaggle_BD /bipolar_dataset.csv
  MULTICLASS_CSV         exists=True  /content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/multi class depression dataset/dataset.csv
  ORIGINAL_RESULTS_DIR   exists=True  /content/drive/MyDrive/Research on BD NYU/multi_source_results


## Part A — Exact reported arrays

**A1** re-scores the saved models to confirm these are the arrays behind 75.6 / 82.3 / 82.6.
**A2** is the key test. The original Check A shuffled physio *across all rows*. Because rows were paired only by label,
a *within-class* shuffle gives a dataset with the same distribution as the "aligned" one.
If within-class F1 ≈ aligned F1, the pairing carries no cross-modal information, and the "alignment" claim is vacuous.
**A3** splits the 34-D physio vector into its sources (WESAD 26 · OBF actigraphy 4 · OBF clinical scores 4).
This assumes the original column order, which is checked.
**A4** checks for duplicate rows across the train/test split.

In [10]:
#@title A · Exact-array diagnostics
npz_path = f"{PATHS['ORIGINAL_RESULTS_DIR']}/validation_data.npz"
A = {}
if not os.path.exists(npz_path):
    print(f'Not found: {npz_path}\nSkipping Part A. Part B rebuilds the arrays from raw data.')
else:
    d = np.load(npz_path)
    T, P, y = d['text_aligned'], d['physio_aligned'], d['labels_aligned'].astype(int)
    X = np.hstack([T, P])
    A['shapes'] = {'text': T.shape, 'physio': P.shape, 'n': len(y), 'class_counts': np.bincount(y).tolist()}
    print('Shapes:', A['shapes'])

    # A1: re-score the saved models on the saved test sets
    try:
        rep = {}
        for name, mfile, xk, yk in [('text', 'text_model.pkl', 'X_test_text', 'y_test_text'),
                                    ('physio', 'physio_model.pkl', 'X_test_physio', 'y_test_physio'),
                                    ('fusion', 'fusion_model.pkl', 'X_test_fusion', 'y_test_fusion')]:
            with open(f"{PATHS['ORIGINAL_RESULTS_DIR']}/{mfile}", 'rb') as f:
                m = pickle.load(f)
            rep[name] = macro_f1(d[yk].astype(int), m.predict(d[xk]))
        A['A1_saved_models_f1'] = rep
        print('A1 saved-model F1 (reported 0.756 / 0.823 / 0.826):', {k: round(v, 3) for k, v in rep.items()})
    except Exception as e:
        print('A1 skipped:', e)

    # Does refitting on the aligned arrays with the original split reproduce the fusion number?
    A['A1_refit_fusion_split_f1'] = split_f1(X, y)
    print('A1 refit fusion F1 on original split:', round(A['A1_refit_fusion_split_f1'], 3))

    # A2: shuffle controls, single original split (same protocol as Check A)
    rng = np.random.default_rng(SEED)
    aligned = [split_f1(X, y, SEED + i) for i in range(N_SHUFFLES)]
    across = [split_f1(np.hstack([T, P[perm_index(len(y), rng)]]), y, SEED + i) for i in range(N_SHUFFLES)]
    within = [split_f1(np.hstack([T, P[perm_index(len(y), rng, y)]]), y, SEED + i) for i in range(N_SHUFFLES)]
    A['A2_split'] = {'aligned': (np.mean(aligned), np.std(aligned)),
                     'across_class_shuffle': (np.mean(across), np.std(across)),
                     'within_class_shuffle': (np.mean(within), np.std(within))}
    # same comparison under repeated 5-fold CV (less split noise)
    rng = np.random.default_rng(SEED + 1)
    cv_al = cv_f1(X, y)
    cv_ac = [cv_f1(np.hstack([T, P[perm_index(len(y), rng)]]), y, n_repeats=1, seed=SEED + i)[0] for i in range(N_SHUFFLES)]
    cv_wi = [cv_f1(np.hstack([T, P[perm_index(len(y), rng, y)]]), y, n_repeats=1, seed=SEED + i)[0] for i in range(N_SHUFFLES)]
    A['A2_cv'] = {'aligned': cv_al, 'across_class_shuffle': (np.mean(cv_ac), np.std(cv_ac)),
                  'within_class_shuffle': (np.mean(cv_wi), np.std(cv_wi))}
    print('\nA2 shuffle controls (macro-F1, mean ± sd)')
    print(f"{'':24s}{'original split':>20s}{'repeated 5-fold CV':>22s}")
    for k in ['aligned', 'across_class_shuffle', 'within_class_shuffle']:
        print(f'{k:24s}{fmt(*A["A2_split"][k]):>20s}{fmt(*A["A2_cv"][k]):>22s}')

    # A3: feature-group ablation (assumes original column order: 26 WESAD, then OBF mean, sd, pctZeros, median, madrs, hads_d, asrs, mdq_pos)
    if P.shape[1] == 34:
        G = {'text_only': T, 'wesad_only': P[:, :26], 'obf_actigraphy_only': P[:, 26:30],
             'obf_clinical_scores_only': P[:, 30:34], 'physio_all': P,
             'physio_without_clinical_scores': P[:, :30],
             'fusion_all': X, 'fusion_without_clinical_scores': np.hstack([T, P[:, :30]])}
        A['A3_feature_groups'] = {}
        print('\nA3 feature-group ablation (macro-F1)')
        print(f"{'':34s}{'original split':>16s}{'repeated 5-fold CV':>22s}")
        for k, M in G.items():
            r = {'split': split_f1(M, y), 'cv': cv_f1(M, y)}
            A['A3_feature_groups'][k] = r
            print(f"{k:34s}{r['split']:>16.3f}{fmt(*r['cv']):>22s}")
    else:
        print(f'\nA3 skipped: physio has {P.shape[1]} columns, expected 34.')

    # A4: exact duplicates across the original split
    tr, te = original_split(y)
    def dup_frac(M):
        train_rows = {r.tobytes() for r in np.round(M[tr], 6)}
        return float(np.mean([r.tobytes() in train_rows for r in np.round(M[te], 6)]))
    A['A4_test_rows_with_exact_duplicate_in_train'] = {'text': dup_frac(T), 'physio': dup_frac(P),
                                                       'wesad_block': dup_frac(P[:, :26]) if P.shape[1] == 34 else None}
    print('\nA4 fraction of test rows with an exact duplicate in train:', A['A4_test_rows_with_exact_duplicate_in_train'])
RESULTS['A'] = A
save_results()

Shapes: {'text': (141, 768), 'physio': (141, 34), 'n': 141, 'class_counts': [47, 47, 47]}
A1 saved-model F1 (reported 0.756 / 0.823 / 0.826): {'text': 0.756, 'physio': 0.823, 'fusion': 0.826}
A1 refit fusion F1 on original split: 0.826

A2 shuffle controls (macro-F1, mean ± sd)
                              original split    repeated 5-fold CV
aligned                        0.854 ± 0.030         0.828 ± 0.012
across_class_shuffle           0.781 ± 0.025         0.752 ± 0.013
within_class_shuffle           0.855 ± 0.030         0.828 ± 0.016

A3 feature-group ablation (macro-F1)
                                    original split    repeated 5-fold CV
text_only                                    0.828         0.736 ± 0.019
wesad_only                                   0.658         0.650 ± 0.024
obf_actigraphy_only                          0.791         0.749 ± 0.011
obf_clinical_scores_only                     0.566         0.700 ± 0.013
physio_all                                   0.827

## Part B — Rebuild from raw data, with provenance
### B0 · Inventory

In [11]:
#@title B0 · Dataset inventory
for k in ['WESAD', 'OBF']:
    p = PATHS[k]
    if os.path.isdir(p):
        entries = os.listdir(p)
        print(f'{k}: {len(entries)} entries, e.g. {sorted(entries)[:12]}')
    else:
        print(f'{k}: MISSING ({p})')
for k in ['KAGGLE_BD_CSV', 'MULTICLASS_CSV']:
    p = PATHS[k]
    print(f'{k}: ' + (f'{os.path.getsize(p)/1e6:.1f} MB' if os.path.exists(p) else f'MISSING ({p})'))

WESAD: 16 entries, e.g. ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6']
OBF: 12 entries, e.g. ['README.txt', 'adhd', 'adhd-info.csv', 'clinical', 'clinical-info.csv', 'control', 'control-info.csv', 'depression', 'depression-info.csv', 'features.csv', 'schizophrenia', 'schizophrenia-info.csv']
KAGGLE_BD_CSV: 3.7 MB
MULTICLASS_CSV: 2.8 MB


### B1 · WESAD: label-code audit and feature extraction
WESAD codebook, from the dataset's readme (**please confirm against `wesad_readme.pdf` in your copy**):
0 = transient/undefined, 1 = baseline, 2 = stress, 3 = amusement, 4 = meditation, 5–7 = ignore.
The original `map_to_bipolar_state()` maps codes {0,3}→Euthymia, {1,6}→Depression, {2,4}→Mania.
The audit below shows which real conditions end up in each "BD state".

In [12]:
#@title B1 · WESAD
WESAD_CODEBOOK = {0: 'transient', 1: 'baseline', 2: 'stress', 3: 'amusement', 4: 'meditation',
                  5: 'ignore_5', 6: 'ignore_6', 7: 'ignore_7'}
ORIGINAL_WESAD_MAP = {0: 0, 3: 0, 1: 1, 6: 1, 2: 2, 4: 2}   # exactly what map_to_bipolar_state() does

def quick_wesad_dataframe(subject_data, window_sec=1):
    """Verbatim feature extraction from multi_source_fusion.ipynb (26 features, 1-s windows)."""
    fs_chest, fs_wrist_bvp, fs_wrist_eda, fs_wrist_acc = 700, 64, 4, 32
    chest = subject_data['signal']['chest']; wrist = subject_data['signal']['wrist']; labels = subject_data['label']
    n_windows = len(chest['ECG']) // (fs_chest * window_sec)
    features = {}
    for s in ['ECG', 'EDA', 'EMG', 'Temp', 'Resp']:
        if s in chest:
            sig = chest[s][:n_windows * fs_chest].reshape((n_windows, fs_chest * window_sec))
            features[f'chest_{s.lower()}_mean'] = sig.mean(1); features[f'chest_{s.lower()}_std'] = sig.std(1)
    if 'ACC' in chest:
        acc = chest['ACC'][:n_windows * fs_chest].reshape((n_windows, fs_chest * window_sec, 3))
        for i, a in enumerate('xyz'):
            features[f'chest_acc_{a}_mean'] = acc[:, :, i].mean(1); features[f'chest_acc_{a}_std'] = acc[:, :, i].std(1)
    if 'BVP' in wrist:
        bvp = wrist['BVP'][:n_windows * fs_wrist_bvp * window_sec].reshape((n_windows, fs_wrist_bvp * window_sec))
        features['wrist_bvp_mean'] = bvp.mean(1); features['wrist_bvp_std'] = bvp.std(1)
    if 'EDA' in wrist:
        eda = wrist['EDA'][:n_windows * fs_wrist_eda * window_sec].reshape((n_windows, fs_wrist_eda * window_sec))
        features['wrist_eda_mean'] = eda.mean(1); features['wrist_eda_std'] = eda.std(1)
    if 'ACC' in wrist:
        acc = wrist['ACC'][:n_windows * fs_wrist_acc * window_sec].reshape((n_windows, fs_wrist_acc * window_sec, 3))
        for i, a in enumerate('xyz'):
            features[f'wrist_acc_{a}_mean'] = acc[:, :, i].mean(1); features[f'wrist_acc_{a}_std'] = acc[:, :, i].std(1)
    ls = labels[:n_windows * fs_chest].reshape((n_windows, fs_chest * window_sec))
    features['label'] = np.array([np.bincount(w.astype(int)).argmax() for w in ls])
    return pd.DataFrame(features)

wesad_cache = f'{CACHE_DIR}/wesad_windows.pkl'
if os.path.exists(wesad_cache):
    wesad_all, wesad_seconds = pd.read_pickle(wesad_cache)
    print('Loaded cached WESAD windows:', wesad_all.shape)
else:
    frames, sec_rows = [], []
    for sid in range(2, 18):    # S2–S17, same loop as the original
        path = f"{PATHS['WESAD']}/S{sid}/S{sid}.pkl"
        try:
            with open(path, 'rb') as f:
                data = pickle.load(f, encoding='latin1')
            lab = np.asarray(data['label']).astype(int)
            sec_rows.append({'subject': f'S{sid}', **{WESAD_CODEBOOK[c]: round(float((lab == c).sum()) / 700, 1)
                                                      for c in range(8)}})
            df = quick_wesad_dataframe(data)
            df['subject'] = f'S{sid}'; df['window'] = np.arange(len(df))
            frames.append(df); print(f'  S{sid}: {len(df)} windows')
            del data
        except Exception as e:
            print(f'  S{sid}: skipped ({e})')

    if len(frames) == 0:
        print('\n[WARNING] CONFIG NEEDED: No WESAD datasets could be found or loaded. Please check your PATHS config in cell 0.')
        wesad_all = pd.DataFrame(columns=['label', 'subject', 'window', 'condition', 'bipolar_state'])
        wesad_seconds = pd.DataFrame()
    else:
        wesad_all = pd.concat(frames, ignore_index=True)
        wesad_seconds = pd.DataFrame(sec_rows)
        pd.to_pickle((wesad_all, wesad_seconds), wesad_cache)

if len(wesad_all) > 0:
    WESAD_FEATURES = [c for c in wesad_all.columns if c not in ('label', 'subject', 'window')]
    assert len(WESAD_FEATURES) == 26, f'Expected 26 WESAD features, got {len(WESAD_FEATURES)}'

    print('\nSeconds of each raw WESAD code per subject:')
    print(wesad_seconds.to_string(index=False))
    wesad_all['condition'] = wesad_all['label'].map(WESAD_CODEBOOK)
    wesad_all['bipolar_state'] = wesad_all['label'].map(ORIGINAL_WESAD_MAP)
    xt = pd.crosstab(wesad_all['condition'], wesad_all['bipolar_state'].map(dict(enumerate(CLASS_NAMES))).fillna('dropped'))
    print('\nWindows: WESAD condition (codebook) × assigned "BD state" (original mapping):')
    print(xt)
    RESULTS['B1_wesad'] = {'n_subjects': wesad_all['subject'].nunique(), 'n_windows': len(wesad_all),
                           'condition_x_state': xt.to_dict()}
    save_results()
else:
    print('\nSkipped calculations: No WESAD data loaded.')

  S2: 6079 windows
  S3: 6493 windows
  S4: 6423 windows
  S5: 6258 windows
  S6: skipped ([Errno 2] No such file or directory: '/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD/S6/S6.pkl')
  S7: skipped ([Errno 2] No such file or directory: '/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD/S7/S7.pkl')
  S8: skipped ([Errno 2] No such file or directory: '/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD/S8/S8.pkl')
  S9: skipped ([Errno 2] No such file or directory: '/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD/S9/S9.pkl')
  S10: 5496 windows
  S11: 5233 windows
  S12: skipped ([Errno 2] No such file or directory: '/content/drive/MyDrive/Research on BD NYU /development/final_modelling/datasets/raw/WESAD/WESAD/S12/S12.pkl')
  S13: 5537 windows
  S14: 5548 windows
  S15: 5252 windows
  S16: 5631 wind

### B2 · OBF-Psychiatric: replicate the cluster labels, then test circularity
The original pipeline concatenated every CSV in the OBF folder, imputed missing values with column means,
and labeled rows by GMM clusters of autoencoder latents of 8 features, **4 of which are clinical rating scales**.
Those same 8 features were then used as model inputs. The checks below:
(i) inspect what the concatenation actually produced (including rows that were *entirely* imputed);
(ii) test whether the cluster labels are recoverable from the clinical scores alone;
(iii) compare cluster labels against the real diagnoses already in the dataset.

In [20]:
#@title B2 · OBF-Psychiatric
OBF_FEATURES = ['mean', 'sd', 'pctZeros', 'median', 'madrs', 'hads_d', 'asrs', 'mdq_pos']
OBF_ACTIGRAPHY, OBF_CLINICAL = OBF_FEATURES[:4], OBF_FEATURES[4:]

frames = []
for fn in os.listdir(PATHS['OBF']):           # unsorted os.listdir, as in the original (row order affects the split)
    fp = os.path.join(PATHS['OBF'], fn)
    if os.path.isfile(fp) and fn.endswith('.csv'):
        try:
            df = pd.read_csv(fp); df['__source_file'] = fn; frames.append(df)
        except Exception as e:
            print('Error reading', fn, e)
obf_raw = pd.concat(frames, ignore_index=True)
print(f'OBF: {len(frames)} CSV files, {len(obf_raw)} rows')
for fn, g in obf_raw.groupby('__source_file'):
    cols = [c for c in g.columns if g[c].notna().any() and c != '__source_file']
    print(f'  {fn}: {len(g)} rows | columns: {cols[:15]}{" ..." if len(cols) > 15 else ""}')

missing = [c for c in OBF_FEATURES if c not in obf_raw.columns]
if missing:
    raise ValueError(f'CONFIG NEEDED: OBF columns {missing} not found. Columns are: {list(obf_raw.columns)}')

def _detect(cands):
    low = {c.lower(): c for c in obf_raw.columns}
    return next((low[c] for c in cands if c in low), None)
OBF_ID_COL = OBF_ID_COL or _detect(['number', 'id', 'user', 'userid', 'patient', 'subject', 'participant'])
OBF_DIAG_COL = OBF_DIAG_COL or _detect(['diagnosis', 'diag', 'group', 'class', 'label', 'condition', 'category'])
print(f'\nOBF_ID_COL = {OBF_ID_COL!r}   OBF_DIAG_COL = {OBF_DIAG_COL!r}  (set in cell 0 if wrong)')

def _col_or_missing(col, unique_missing=False):
    """Column as strings; missing values (e.g. rows from CSVs with a different schema) made explicit."""
    v = obf_raw[col].astype(object)
    miss = v.isna().values
    out = v.where(~v.isna(), '<missing>').astype(str).values
    if unique_missing:   # each ID-less row becomes its own group so grouped CV stays valid
        out = np.where(miss, np.array([f'<missing_{i}>' for i in range(len(v))]), out)
    return out, miss
if OBF_ID_COL:
    obf_person, id_missing = _col_or_missing(OBF_ID_COL, unique_missing=True)
    print(f'Rows with no {OBF_ID_COL!r} value: {int(id_missing.sum())}')
else:
    obf_person = np.array([f'row_{i}' for i in range(len(obf_raw))])

all_nan = obf_raw[OBF_FEATURES].isna().all(axis=1)
some_nan = obf_raw[OBF_FEATURES].isna().any(axis=1)
print(f'Rows with ALL 8 features missing (fully imputed with column means): {int(all_nan.sum())} / {len(obf_raw)}')
print(f'Rows with ANY feature missing: {int(some_nan.sum())}')
print('Missing rate per feature:', obf_raw[OBF_FEATURES].isna().mean().round(3).to_dict())

# ---- Replicate the original labeling (cells 7–10 of multi_source_fusion.ipynb) ----
X_obf = obf_raw[OBF_FEATURES].fillna(obf_raw[OBF_FEATURES].mean())
obf_tr, obf_te = train_test_split(np.arange(len(X_obf)), test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_obf.values[obf_tr])
obf_scaled = scaler.transform(X_obf.values)          # same values as the original's per-split transform

if SMOKE_TEST:   # torch unavailable in the test sandbox: PCA stand-in for the autoencoder
    from sklearn.decomposition import PCA
    pca = PCA(3, random_state=42).fit(obf_scaled[obf_tr]); Z = pca.transform(obf_scaled)
else:
    import torch, torch.nn as nn
    torch.manual_seed(42)   # the original did not seed torch, so cluster labels may differ slightly from the reported run
    class Autoencoder(nn.Module):
        def __init__(self, d=8, k=3):
            super().__init__()
            self.encoder = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, k), nn.ReLU())
            self.decoder = nn.Sequential(nn.Linear(k, 64), nn.ReLU(), nn.Linear(64, d))
        def forward(self, x): return self.decoder(self.encoder(x))
    ae = Autoencoder(obf_scaled.shape[1]); opt = torch.optim.Adam(ae.parameters(), lr=1e-3); lossf = nn.MSELoss()
    Xt = torch.FloatTensor(obf_scaled[obf_tr])
    for epoch in range(50):
        for i in range(0, len(Xt), 32):
            b = Xt[i:i + 32]; opt.zero_grad(); l = lossf(ae(b), b); l.backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        Z = ae.encoder(torch.FloatTensor(obf_scaled)).numpy()

from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=3, random_state=42).fit(Z[obf_tr])
obf_labels = gmm.predict(Z)                            # cluster_to_state was the identity {0:0, 1:1, 2:2}
print('\nGMM "BD state" counts:', np.bincount(obf_labels, minlength=3).tolist())
print('Mean raw clinical scores per cluster (this is how clusters were named):')
print(obf_raw.assign(cluster=obf_labels).groupby('cluster')[OBF_CLINICAL].mean().round(2))

# ---- Circularity: can each feature group recover the cluster labels? ----
B2 = {'n_rows': len(obf_raw), 'n_fully_imputed_rows': int(all_nan.sum()),
      'gmm_counts': np.bincount(obf_labels, minlength=3).tolist()}
yl = obf_labels.astype(int)
grp = obf_person if OBF_ID_COL else None
print('\nRecovering the GMM labels from feature subsets (macro-F1, repeated 5-fold CV):')
for name, cols in [('clinical_scores_only', [4, 5, 6, 7]), ('actigraphy_only', [0, 1, 2, 3]), ('all_8', list(range(8)))]:
    r = cv_f1(obf_scaled[:, cols], yl)
    rg = cv_f1(obf_scaled[:, cols], yl, groups=grp) if grp is not None else None
    B2[f'recover_gmm_from_{name}'] = {'row_cv': r, 'subject_grouped_cv': rg}
    print(f'  {name:22s} row-CV {fmt(*r)}' + (f' | subject-grouped {fmt(*rg)}' if rg else ''))

if OBF_DIAG_COL:
    diag, diag_missing = _col_or_missing(OBF_DIAG_COL)
    ct = pd.crosstab(diag, obf_labels)
    print(f'\nReal diagnosis ({OBF_DIAG_COL}) × GMM "BD state" (0=Euthymia, 1=Depression, 2=Mania):')
    print(ct)
    B2['diagnosis_x_gmm'] = ct.to_dict(); B2['ARI_diagnosis_vs_gmm'] = adjusted_rand_score(diag, obf_labels)
    print('ARI(diagnosis, GMM labels):', round(B2['ARI_diagnosis_vs_gmm'], 3))
    if grp is not None:   # the honest version of the task this dataset supports
        keep = ~diag_missing
        dy = pd.factorize(diag[keep])[0]
        if len(np.unique(dy)) > 1 and np.bincount(dy).min() >= 2:
            B2['diagnosis_from_actigraphy_subject_grouped'] = cv_f1(obf_scaled[keep][:, :4], dy, groups=grp[keep])
            print('Real diagnosis from actigraphy only, subject-grouped CV:',
                  fmt(*B2['diagnosis_from_actigraphy_subject_grouped']))
else:
    print('\nNo diagnosis column detected; set OBF_DIAG_COL in cell 0 to run the diagnosis comparison.')
RESULTS['B2_obf'] = B2
save_results()

OBF: 6 CSV files, 1059 rows
  adhd-info.csv: 45 rows | columns: ['number', 'days', 'gender', 'age', 'acc_time', 'adhd', 'add', 'bipolar', 'unipolar', 'anxiety', 'substance', 'other', 'ct', 'mdq_pos', 'wurs'] ...
  clinical-info.csv: 40 rows | columns: ['number', 'days', 'gender', 'age', 'acc_time', 'adhd', 'add', 'bipolar', 'unipolar', 'anxiety', 'substance', 'other', 'ct', 'mdq_pos', 'wurs'] ...
  control-info.csv: 32 rows | columns: ['number', 'days', 'gender', 'age', 'acc_time']
  depression-info.csv: 23 rows | columns: ['number', 'days', 'gender', 'age', 'acc_time', 'afftype', 'melanch', 'inpatient', 'marriage', 'work', 'madrs1', 'madrs2', 'migraine']
  features.csv: 897 rows | columns: ['user', 'mean', 'sd', 'pctZeros', 'median', 'q25', 'q75', 'class']
  schizophrenia-info.csv: 22 rows | columns: ['number', 'days', 'gender', 'age', 'acc_time', 'migraine', 'schtype', 'bprs', 'cloz', 'antipsych', 'moodst']

OBF_ID_COL = 'number'   OBF_DIAG_COL = 'class'  (set in cell 0 if wrong)
Row

### B3 · Text: replicate labels, then test circularity and source confounding
Kaggle BD labels were KMeans clusters of the *same* BERT embeddings used as model input, named by VADER sentiment.
Multi-class labels were mapped by hand ("bipolar"→Depression, "psychotic"→Mania).
We also test whether the label is confounded with *which corpus* a post came from.

In [14]:
#@title B3 · Text corpora
def clean_text(text):
    text = re.sub(r'http\S+', '', str(text)); text = re.sub(r'[^a-zA-Z\s]', '', text); return text.lower()

kag = pd.read_csv(PATHS['KAGGLE_BD_CSV'], encoding='latin1', delimiter=';')
kag['post'] = kag['post'].apply(clean_text)
mc = pd.read_csv(PATHS['MULTICLASS_CSV'])
low = {c.lower(): c for c in mc.columns}
text_col = next(low[c] for c in ['tweet', 'tweets', 'text'] if c in low)
label_col = next(low[c] for c in ['label', 'labels', 'class'] if c in low)
mc[text_col] = mc[text_col].apply(clean_text)
LABEL_MAP = {'major depressive': 1, 'atypical': 1, 'psychotic': 2, 'no': 0, 'bipolar': 1, 'postpartum': -1}
print('Multi-class raw label values:', mc[label_col].value_counts().to_dict())
mc['bd_label'] = mc[label_col].map(LABEL_MAP)
unmapped = mc['bd_label'].isna().sum()
mc = mc[mc['bd_label'] != -1].dropna(subset=['bd_label'])
print(f'Rows silently dropped as unmapped: {unmapped}; kept: {len(mc)}')

def embed(texts, cache_name):
    path = f'{CACHE_DIR}/{cache_name}.npy'
    if os.path.exists(path):
        E = np.load(path)
        if len(E) == len(texts): return E
    if SMOKE_TEST:
        E = np.random.default_rng(0).normal(size=(len(texts), 768)).astype(np.float32)
    else:
        import torch
        from transformers import BertTokenizer, BertModel
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        tok = BertTokenizer.from_pretrained('bert-base-uncased'); bert = BertModel.from_pretrained('bert-base-uncased').to(dev).eval()
        out = []
        for i in range(0, len(texts), BERT_BATCH):
            enc = tok(list(texts[i:i + BERT_BATCH]), return_tensors='pt', truncation=True, max_length=512, padding=True).to(dev)
            with torch.no_grad():
                h = bert(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())   # masked mean == original unpadded mean
            if (i // BERT_BATCH) % 50 == 0: print(f'  {cache_name}: {i}/{len(texts)}')
        E = np.vstack(out)
    np.save(path, E); return E

E_kag = embed(kag['post'].values, 'emb_kaggle_bd')
E_mc = embed(mc[text_col].values, 'emb_multiclass')

from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, random_state=42).fit(E_kag)   # NB: sklearn's n_init default changed in 1.4; cluster IDs may differ from the original run
kag_labels = km.labels_                                  # cluster_to_label was the identity
B3 = {'kaggle_cluster_counts': np.bincount(kag_labels, minlength=3).tolist()}
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    va = SentimentIntensityAnalyzer()
    sent = np.array([va.polarity_scores(t)['compound'] for t in kag['post'].values])
    B3['vader_mean_by_cluster'] = {int(c): float(sent[kag_labels == c].mean()) for c in range(3)}
    print('Mean VADER sentiment per KMeans cluster (basis for naming):', B3['vader_mean_by_cluster'])
except ImportError:
    print('(pip install vaderSentiment to reproduce the sentiment-based naming step)')

text_emb = np.vstack([E_kag, E_mc]).astype(np.float32)
text_labels = np.concatenate([kag_labels, mc['bd_label'].astype(int).values])
text_source = np.array(['kaggle_bd_reddit'] * len(E_kag) + ['multiclass_twitter'] * len(E_mc))

# Circularity: the Kaggle labels are a deterministic function of the input features
B3['kaggle_labels_recovered_by_nearest_centroid'] = float((km.predict(E_kag) == kag_labels).mean())
print('Kaggle labels recovered exactly by nearest KMeans centroid:', B3['kaggle_labels_recovered_by_nearest_centroid'])

# Source confounding
ct = pd.crosstab(text_source, pd.Series(text_labels).map(dict(enumerate(CLASS_NAMES))).values)
print('\nCorpus × label:'); print(ct)
B3['source_x_label'] = ct.to_dict()
rng = np.random.default_rng(SEED)
sub = np.concatenate([rng.choice(np.where(text_source == s)[0], min(2000, (text_source == s).sum()), replace=False)
                      for s in np.unique(text_source)])
src_y = pd.factorize(text_source[sub])[0]
B3['source_identity_probe_cv'] = cv_f1(text_emb[sub], src_y, n_repeats=1)
print('Corpus-identity probe (predict Reddit vs Twitter from embeddings), macro-F1:', fmt(*B3['source_identity_probe_cv']))
maj = pd.Series(text_labels).groupby(text_source).agg(lambda s: s.value_counts().idxmax())
B3['label_from_source_only_f1'] = macro_f1(text_labels, maj.loc[text_source].values)
print('Macro-F1 from predicting each corpus\'s majority label (no text used):', round(B3['label_from_source_only_f1'], 3))
RESULTS['B3_text'] = B3
save_results()

Multi-class raw label values: {'postpartum': 3747, 'major depressive': 2517, 'bipolar': 2443, 'psychotic': 2312, 'no': 1985, 'atypical': 1980}
Rows silently dropped as unmapped: 12; kept: 11237
(pip install vaderSentiment to reproduce the sentiment-based naming step)
Kaggle labels recovered exactly by nearest KMeans centroid: 1.0

Corpus × label:
col_0               Depression  Euthymia  Mania
row_0                                          
kaggle_bd_reddit           855       374    463
multiclass_twitter        6940      1985   2312
Corpus-identity probe (predict Reddit vs Twitter from embeddings), macro-F1: 0.936 ± 0.000
Macro-F1 from predicting each corpus's majority label (no text used): 0.251


### B4 · Rebuild the fused dataset with provenance, and replicate the reported numbers
Same steps and seeds as the original. Each fused row also carries its WESAD subject/window, its OBF row/person,
and its text corpus. `build_fused(seed)` lets us redraw the label-conditional pairing with other seeds.

In [21]:
#@title B4 · Rebuild & replicate
from sklearn.utils import resample

wc = wesad_all.dropna(subset=['bipolar_state']).reset_index(drop=True)
wc['bipolar_state'] = wc['bipolar_state'].astype(int); wc['row_id'] = np.arange(len(wc))

if len(wc) == 0:
    print('[WARNING] Cannot balance WESAD because wesad_all is empty. Creating an empty placeholder.')
    wesad_balanced = pd.DataFrame(columns=list(wesad_all.columns) + ['row_id'])
    dup_rate = 0.0
else:
    min_count = wc['bipolar_state'].value_counts().min()
    wesad_balanced = pd.concat([resample(wc[wc['bipolar_state'] == c], n_samples=min_count, random_state=42)   # replace=True (sklearn default)
                                for c in [0, 1, 2]]).sample(frac=1, random_state=42).reset_index(drop=True)
    dup_rate = 1 - wesad_balanced['row_id'].nunique() / len(wesad_balanced)
    print(f'WESAD balanced: {len(wesad_balanced)} rows; {dup_rate:.1%} are duplicates created by resampling with replacement')

obf_df = pd.DataFrame(obf_scaled, columns=[f'obf_feat_{i}' for i in range(8)])
obf_df['bipolar_state'] = obf_labels
obf_df['obf_person'] = obf_person

def build_fused(seed=42):
    """Cells 11 and 17 of the original, with metadata carried along. seed=42 reproduces the original draw."""
    if len(wesad_balanced) == 0:
        # Return empty dictionary or basic structure to prevent cascading crashes
        return {
            'T': np.empty((0, 768)), 'P': np.empty((0, 34)), 'y': np.array([], dtype=int),
            'meta': pd.DataFrame(columns=['label', 'text_source', 'wesad_subject', 'wesad_window', 'wesad_row_id', 'obf_row', 'obf_person']),
            'physio_full': np.empty((0, 34)), 'physio_labels': np.array([])
        }
    np.random.seed(seed)
    wl = {c: wesad_balanced.index[wesad_balanced['bipolar_state'] == c].values for c in range(3)}
    ol = {c: obf_df.index[obf_df['bipolar_state'] == c].values for c in range(3)}
    mpc = min(min(len(wl[c]) for c in range(3)), min(len(ol[c]) for c in range(3)))
    wi, oi, lab = [], [], []
    for c in range(3):
        wi.extend(np.random.choice(wl[c], mpc, replace=False)); oi.extend(np.random.choice(ol[c], mpc, replace=False)); lab.extend([c] * mpc)
    wi, oi, lab = map(np.array, (wi, oi, lab)); s = np.random.permutation(len(wi)); wi, oi, lab = wi[s], oi[s], lab[s]
    physio = np.hstack([wesad_balanced.loc[wi, WESAD_FEATURES].values, obf_df.loc[oi, [f'obf_feat_{i}' for i in range(8)]].values])
    # cell 17
    np.random.seed(seed)
    ti, pi, l2 = [], [], []
    for c in range(3):
        t_idx = np.where(text_labels == c)[0]; p_idx = np.where(lab == c)[0]; n = min(len(t_idx), len(p_idx))
        ti.extend(np.random.choice(t_idx, n, replace=False)); pi.extend(np.random.choice(p_idx, n, replace=False)); l2.extend([c] * n)
    ti, pi, l2 = map(np.array, (ti, pi, l2)); s = np.random.permutation(len(ti)); ti, pi, l2 = ti[s], pi[s], l2[s]
    meta = pd.DataFrame({'label': l2, 'text_source': text_source[ti],
                         'wesad_subject': wesad_balanced.loc[wi[pi], 'subject'].values,
                         'wesad_window': wesad_balanced.loc[wi[pi], 'window'].values,
                         'wesad_row_id': wesad_balanced.loc[wi[pi], 'row_id'].values,
                         'obf_row': oi[pi], 'obf_person': obf_df.loc[oi[pi], 'obf_person'].values})
    return {'T': text_emb[ti], 'P': physio[pi], 'y': l2.astype(int), 'meta': meta,
            'physio_full': physio, 'physio_labels': lab}

F = build_fused(42)
print('Fused rows:', len(F['y']), '| per class:', np.bincount(F['y']).tolist() if len(F['y']) > 0 else [],
      '| WESAD subjects:', F['meta']['wesad_subject'].nunique(), '| corpora:', F['meta']['text_source'].value_counts().to_dict())

tr_t, te_t = train_test_split(np.arange(len(text_labels)), test_size=0.2, stratify=text_labels, random_state=42)
if len(F['y']) > 0:
    rep = {'text_only_full_corpus': macro_f1(text_labels[te_t], rf().fit(text_emb[tr_t], text_labels[tr_t]).predict(text_emb[te_t])),
           'physio_only': split_f1(F['physio_full'], F['physio_labels']),
           'fusion': split_f1(np.hstack([F['T'], F['P']]), F['y'])}
    print('Replication on original splits (reported 0.756 / 0.823 / 0.826):', {k: round(v, 3) for k, v in rep.items()})
else:
    rep = {'text_only_full_corpus': 0.0, 'physio_only': 0.0, 'fusion': 0.0}
    print('Replication skipped because no fused rows are available.')

RESULTS['B4_replication'] = {'wesad_duplicate_rate': dup_rate, 'n_fused': len(F['y']), **rep}
save_results()

WESAD balanced: 40437 rows; 29.3% are duplicates created by resampling with replacement
Fused rows: 141 | per class: [47, 47, 47] | WESAD subjects: 11 | corpora: {'multiclass_twitter': 129, 'kaggle_bd_reddit': 12}
Replication on original splits (reported 0.756 / 0.823 / 0.826): {'text_only_full_corpus': 0.756, 'physio_only': 0.823, 'fusion': 0.826}


### B5 · Leakage audit
How "unseen" is the test set? For each fused test row, do we find the same WESAD subject, the same window,
a window less than 5 s away, or the same OBF person in the training set? Then compare a random-window split with
leave-subject-out on WESAD alone, using a larger sample.

In [16]:
#@title B5 · Leakage
m = F['meta']; tr, te = original_split(F['y'])
def frac(cond): return float(np.mean(cond))
tr_m, te_m = m.iloc[tr], m.iloc[te]
near = []
for _, r in te_m.iterrows():
    same = tr_m[tr_m['wesad_subject'] == r['wesad_subject']]
    near.append(len(same) > 0 and (np.abs(same['wesad_window'] - r['wesad_window']) < 5).any())
B5 = {'test_rows_same_wesad_subject_in_train': frac(te_m['wesad_subject'].isin(tr_m['wesad_subject'])),
      'test_rows_same_wesad_window_in_train': frac(te_m['wesad_row_id'].isin(tr_m['wesad_row_id'])),
      'test_rows_wesad_window_within_5s_in_train': frac(near),
      'test_rows_same_obf_person_in_train': frac(te_m['obf_person'].isin(tr_m['obf_person'])) if OBF_ID_COL else None}
for k, v in B5.items(): print(f'{k:45s} {v}')

rng = np.random.default_rng(SEED)
samp = np.concatenate([rng.choice(np.where(wc['bipolar_state'] == c)[0], min(2000, (wc['bipolar_state'] == c).sum()), replace=False) for c in range(3)])
Xw, yw, gw = wc.loc[samp, WESAD_FEATURES].values, wc.loc[samp, 'bipolar_state'].values, wc.loc[samp, 'subject'].values
B5['wesad_random_window_cv'] = cv_f1(Xw, yw, n_repeats=2)
loso = np.empty_like(yw)
for trn, tst in GroupKFold(len(np.unique(gw))).split(Xw, yw, gw):
    loso[tst] = rf().fit(Xw[trn], yw[trn]).predict(Xw[tst])
B5['wesad_leave_one_subject_out'] = macro_f1(yw, loso)
print(f"\nWESAD-only, mapped 3 classes: random-window CV {fmt(*B5['wesad_random_window_cv'])} vs leave-one-subject-out {B5['wesad_leave_one_subject_out']:.3f}")
RESULTS['B5_leakage'] = B5
save_results()

test_rows_same_wesad_subject_in_train         1.0
test_rows_same_wesad_window_in_train          0.0
test_rows_wesad_window_within_5s_in_train     0.0
test_rows_same_obf_person_in_train            0.0

WESAD-only, mapped 3 classes: random-window CV 0.962 ± 0.001 vs leave-one-subject-out 0.424


### B6 · Waterfall
Cumulative corrections, each evaluated over `N_BUILDS` independent draws of the pairing.
At every stage we also report the within-class and across-class shuffle diagnostics and the unimodal baselines.

In [17]:
#@title B6 · Waterfall
CLIN = list(range(30, 34))   # OBF clinical-score columns inside P
def stage_eval(T, P, y, groups, rng, reps):
    X = np.hstack([T, P])
    wi = perm_index(len(y), rng, y); ac = perm_index(len(y), rng)
    g_w = None if groups is None else groups[wi]; g_a = None if groups is None else groups[ac]
    # groups follow the physio rows they belong to
    return {'fusion': cv_f1(X, y, groups, reps)[0], 'text_only': cv_f1(T, y, groups, reps)[0],
            'physio_only': cv_f1(P, y, groups, reps)[0],
            'fusion_within_class_shuffle': cv_f1(np.hstack([T, P[wi]]), y, g_w, reps)[0],
            'fusion_across_class_shuffle': cv_f1(np.hstack([T, P[ac]]), y, g_a, reps)[0]}

stages = {'S1 repeated CV (same features)': dict(drop_clin=False, group=None),
          'S2 + clinical scores removed': dict(drop_clin=True, group=None),
          'S3 + grouped by WESAD subject': dict(drop_clin=True, group='wesad_subject')}
if OBF_ID_COL:
    stages['S3b + grouped by OBF person (alt.)'] = dict(drop_clin=True, group='obf_person')

rows = {k: [] for k in stages}
for b in range(N_BUILDS):
    Fb = build_fused(42 + b); rng = np.random.default_rng(1000 + b)
    for name, cfg in stages.items():
        P = np.delete(Fb['P'], CLIN, axis=1) if cfg['drop_clin'] else Fb['P']
        g = Fb['meta'][cfg['group']].values if cfg['group'] else None
        rows[name].append(stage_eval(Fb['T'], P, Fb['y'], g, rng, WATERFALL_CV_REPEATS))
    print(f'  build {b + 1}/{N_BUILDS} done')

W = {'S0 reported protocol (single split, build 42)': {'fusion': rep['fusion']}}
for name, lst in rows.items():
    W[name] = {k: (float(np.mean([r[k] for r in lst])), float(np.std([r[k] for r in lst]))) for k in lst[0]}
cols = ['fusion', 'text_only', 'physio_only', 'fusion_within_class_shuffle', 'fusion_across_class_shuffle']
print(f"\n{'stage':46s}" + ''.join(f'{c[:22]:>24s}' for c in cols))
for name, r in W.items():
    print(f'{name:46s}' + ''.join(f"{(fmt(*r[c]) if isinstance(r.get(c), tuple) else (fmt(r[c]) if c in r else '-')):>24s}" for c in cols))
RESULTS['B6_waterfall'] = W
save_results()

  build 1/10 done
  build 2/10 done
  build 3/10 done
  build 4/10 done
  build 5/10 done
  build 6/10 done
  build 7/10 done
  build 8/10 done
  build 9/10 done
  build 10/10 done

stage                                                           fusion               text_only             physio_only  fusion_within_class_sh  fusion_across_class_sh
S0 reported protocol (single split, build 42)                    0.826                       -                       -                       -                       -
S1 repeated CV (same features)                           0.827 ± 0.025           0.719 ± 0.034           0.879 ± 0.029           0.829 ± 0.020           0.720 ± 0.041
S2 + clinical scores removed                             0.761 ± 0.032           0.719 ± 0.034           0.810 ± 0.035           0.757 ± 0.027           0.719 ± 0.039
S3 + grouped by WESAD subject                            0.740 ± 0.030           0.710 ± 0.033           0.717 ± 0.056           0.739 ± 0.034        

### B7 · Calibration audit
(a) Replicates the original: isotonic calibration fitted on the **training** set of a prefit forest, ECE on 29 test rows,
at several bin counts, with bootstrap CIs. (b) Calibration fitted by internal CV and evaluated on held-out folds over 20 splits.

In [18]:
#@title B7 · Calibration
X = np.hstack([F['T'], F['P']]); y = F['y']; tr, te = original_split(y)
base = rf().fit(X[tr], y[tr])
try:
    cal = CalibratedClassifierCV(base, method='isotonic', cv='prefit').fit(X[tr], y[tr])
except Exception:
    from sklearn.frozen import FrozenEstimator   # sklearn >= 1.6
    cal = CalibratedClassifierCV(FrozenEstimator(base), method='isotonic').fit(X[tr], y[tr])
B7 = {'n_test': len(te), 'original_protocol': {}}
rng = np.random.default_rng(SEED)
for name, model in [('uncalibrated', base), ('isotonic_fit_on_train', cal)]:
    pr = model.predict_proba(X[te])
    for nb in [5, 10, 15]:
        boots = [ece(y[te][i], pr[i], nb) for i in (rng.integers(0, len(te), len(te)) for _ in range(1000))]
        B7['original_protocol'][f'{name}_bins{nb}'] = {'ece': ece(y[te], pr, nb), 'ci95': np.percentile(boots, [2.5, 97.5]).tolist()}
for k, v in B7['original_protocol'].items():
    print(f"{k:32s} ECE {v['ece']:.3f}  95% CI [{v['ci95'][0]:.3f}, {v['ci95'][1]:.3f}]")
heldout = []
for i, (trn, tst) in enumerate(StratifiedShuffleSplit(20, test_size=0.2, random_state=SEED).split(X, y)):
    c = CalibratedClassifierCV(rf(SEED + i), method='isotonic', cv=5).fit(X[trn], y[trn])
    heldout.append(ece(y[tst], c.predict_proba(X[tst]), 10))
B7['heldout_cv_calibrated_ece10'] = {'mean': float(np.mean(heldout)), 'p2.5': float(np.percentile(heldout, 2.5)),
                                     'p97.5': float(np.percentile(heldout, 97.5))}
print('Held-out calibration over 20 splits (10 bins):', B7['heldout_cv_calibrated_ece10'])
RESULTS['B7_calibration'] = B7
save_results()

uncalibrated_bins5               ECE 0.305  95% CI [0.166, 0.411]
uncalibrated_bins10              ECE 0.305  95% CI [0.197, 0.416]
uncalibrated_bins15              ECE 0.305  95% CI [0.193, 0.411]
isotonic_fit_on_train_bins5      ECE 0.048  95% CI [0.027, 0.217]
isotonic_fit_on_train_bins10     ECE 0.104  95% CI [0.057, 0.247]
isotonic_fit_on_train_bins15     ECE 0.178  95% CI [0.099, 0.297]
Held-out calibration over 20 splits (10 bins): {'mean': 0.1306866310880395, 'p2.5': 0.0784413203524989, 'p97.5': 0.18534370133830794}


## Summary: copy everything between the markers back into the chat

In [19]:
#@title Summary
def rnd(o):
    """Round every float in a nested structure to 3 decimals for a compact paste-back."""
    if isinstance(o, dict): return {str(k): rnd(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [rnd(v) for v in o]
    if isinstance(o, (float, np.floating)): return round(float(o), 3)
    if isinstance(o, np.integer): return int(o)
    return o
lines = ['=== PASTE BACK ===', f"run {RESULTS['meta']['started']}  smoke={SMOKE_TEST}"]
for key in ['A', 'B1_wesad', 'B2_obf', 'B3_text', 'B4_replication', 'B5_leakage', 'B6_waterfall', 'B7_calibration']:
    if RESULTS.get(key):
        lines.append(f'{key}: ' + json.dumps(rnd(RESULTS[key]), default=str))
lines.append('=== END ===')
print('\n'.join(lines))
RESULTS['meta']['finished'] = time.strftime('%Y-%m-%d %H:%M'); save_results()
print('\nFull results:', f'{OUT_DIR}/results.json')

=== PASTE BACK ===
run 2026-10-03 16:44  smoke=False
A: {"shapes": {"text": [141, 768], "physio": [141, 34], "n": 141, "class_counts": [47, 47, 47]}, "A1_saved_models_f1": {"text": 0.756, "physio": 0.823, "fusion": 0.826}, "A1_refit_fusion_split_f1": 0.826, "A2_split": {"aligned": [0.854, 0.03], "across_class_shuffle": [0.781, 0.025], "within_class_shuffle": [0.855, 0.03]}, "A2_cv": {"aligned": [0.828, 0.012], "across_class_shuffle": [0.752, 0.013], "within_class_shuffle": [0.828, 0.016]}, "A3_feature_groups": {"text_only": {"split": 0.828, "cv": [0.736, 0.019]}, "wesad_only": {"split": 0.658, "cv": [0.65, 0.024]}, "obf_actigraphy_only": {"split": 0.791, "cv": [0.749, 0.011]}, "obf_clinical_scores_only": {"split": 0.566, "cv": [0.7, 0.013]}, "physio_all": {"split": 0.827, "cv": [0.838, 0.011]}, "physio_without_clinical_scores": {"split": 0.722, "cv": [0.747, 0.018]}, "fusion_all": {"split": 0.826, "cv": [0.828, 0.012]}, "fusion_without_clinical_scores": {"split": 0.724, "cv": [0.766, 0

In [22]:
import numpy as np, pandas as pd
print(pd.crosstab(obf_labels, all_nan.values, rownames=['gmm'], colnames=['all_8_missing']))

OBF_COLS = [f'obf_feat_{i}' for i in range(8)]
blk = np.round(F['P'][:, 26:34], 6)
for c in range(3):
    rows = blk[F['y'] == c]
    uniq = len({r.tobytes() for r in rows})
    print(f'class {c}: {len(rows)} rows, {uniq} unique OBF vectors')

m = F['meta']
imp = set(np.where(all_nan.values)[0])
print('fused rows whose OBF row was fully imputed, by class:',
      pd.crosstab(m['label'], m['obf_row'].isin(imp)).to_dict())

all_8_missing  False  True 
gmm                        
0                822     78
1                112      0
2                 47      0
class 0: 47 rows, 40 unique OBF vectors
class 1: 47 rows, 46 unique OBF vectors
class 2: 47 rows, 47 unique OBF vectors
fused rows whose OBF row was fully imputed, by class: {False: {0: 39, 1: 47, 2: 47}, True: {0: 8, 1: 0, 2: 0}}


In [23]:
print(pd.crosstab(obf_raw['__source_file'], obf_labels))
print(obf_raw.groupby('__source_file')[OBF_FEATURES].apply(lambda g: g.notna().mean()).round(2))

col_0                     0   1   2
__source_file                      
adhd-info.csv             0  18  27
clinical-info.csv         1  20  19
control-info.csv         32   0   0
depression-info.csv      23   0   0
features.csv            822  74   1
schizophrenia-info.csv   22   0   0
                        mean   sd  pctZeros  median  madrs  hads_d  asrs  \
__source_file                                                              
adhd-info.csv            0.0  0.0       0.0     0.0   0.93    0.91  0.98   
clinical-info.csv        0.0  0.0       0.0     0.0   0.95    0.92  0.92   
control-info.csv         0.0  0.0       0.0     0.0   0.00    0.00  0.00   
depression-info.csv      0.0  0.0       0.0     0.0   0.00    0.00  0.00   
features.csv             1.0  1.0       1.0     1.0   0.00    0.00  0.00   
schizophrenia-info.csv   0.0  0.0       0.0     0.0   0.00    0.00  0.00   

                        mdq_pos  
__source_file                    
adhd-info.csv              0.96  
c

In [24]:
# 1. Confirm the frozen-half signature in the fused data
for name, sl in [('actigraphy', slice(26,30)), ('clinical', slice(30,34))]:
    for c in range(3):
        rows = np.round(F['P'][F['y']==c, sl], 6)
        print(f'{name:11s} class {c}: {len(rows)} rows, {len({r.tobytes() for r in rows})} unique')

# 2. Cluster label predicted from the missingness pattern alone — no feature values
mask = obf_raw[OBF_FEATURES].isna().values.astype(int)
print('GMM label from missingness mask only:', cv_f1(mask, obf_labels))

# 3. Does a proper join exist? Check for a participant key and a bipolar/unipolar field
for fn, g in obf_raw.groupby('__source_file'):
    print(fn, [c for c in g.columns if g[c].notna().any()])

actigraphy  class 0: 47 rows, 40 unique
actigraphy  class 1: 47 rows, 32 unique
actigraphy  class 2: 47 rows, 2 unique
clinical    class 0: 47 rows, 1 unique
clinical    class 1: 47 rows, 16 unique
clinical    class 2: 47 rows, 47 unique
GMM label from missingness mask only: (0.542489021658092, 0.005431189817065511)
adhd-info.csv ['__source_file', 'number', 'days', 'gender', 'age', 'acc_time', 'adhd', 'add', 'bipolar', 'unipolar', 'anxiety', 'substance', 'other', 'ct', 'mdq_pos', 'wurs', 'asrs', 'madrs', 'hads_a', 'hads_d', 'med', 'med_antidepr', 'med_anxiety_benzo', 'med_sleep', 'med_stimulants']
clinical-info.csv ['__source_file', 'number', 'days', 'gender', 'age', 'acc_time', 'adhd', 'add', 'bipolar', 'unipolar', 'anxiety', 'substance', 'other', 'ct', 'mdq_pos', 'wurs', 'asrs', 'madrs', 'hads_a', 'hads_d', 'med', 'med_antidepr', 'med_moodstab', 'med_antipsych', 'med_anxiety_benzo', 'med_sleep']
control-info.csv ['__source_file', 'number', 'days', 'gender', 'age', 'acc_time']
depress

In [25]:
feat = obf_raw[obf_raw['__source_file']=='features.csv']
info = obf_raw[obf_raw['__source_file'].str.endswith('-info.csv')]
print(feat['user'].nunique(), len(feat), info['number'].nunique(), len(info))
print(len(set(feat['user'].astype(str)) & set(info['number'].astype(str))))
print(feat['class'].value_counts())

77 897 162 162
77
class
control          366
depression       268
schizophrenia    263
Name: count, dtype: int64


In [26]:
dep = obf_raw[obf_raw['__source_file']=='depression-info.csv']
print(dep['afftype'].value_counts(dropna=False))
m = feat.merge(info[['number','__source_file']], left_on=feat['user'].astype(str),
               right_on=info['number'].astype(str), how='left')
print(m.groupby('__source_file_y')['user'].nunique())
import os; print(os.listdir(PATHS['OBF']))

afftype
2.0    15
1.0     7
3.0     1
Name: count, dtype: int64
__source_file_y
control-info.csv          32
depression-info.csv       23
schizophrenia-info.csv    22
Name: user, dtype: int64
['features.csv', 'control-info.csv', 'adhd-info.csv', 'clinical-info.csv', 'depression-info.csv', 'schizophrenia-info.csv', 'README.txt', 'depression', 'clinical', 'control', 'schizophrenia', 'adhd']


In [27]:
print(open(f"{PATHS['OBF']}/README.txt").read())
for d in ['adhd','clinical','depression','control','schizophrenia']:
    p = f"{PATHS['OBF']}/{d}"
    fs = os.listdir(p)
    print(d, len(fs), fs[:3])
print(pd.read_csv(f"{PATHS['OBF']}/adhd/{sorted(os.listdir(PATHS['OBF']+'/adhd'))[0]}").head())


#######
Dataset
#######

OBF-Psychiatric, a motor activity dataset of patients diagnosed with major depression, schizophrenia, and ADHD

#######
Authors
#######

Enrique Garcia-Ceja (enrique.gc@tec.mx), Andrea Stautland, Michael A. Riegler (michael@simula.no), Pål Halvorsen, Salvador Hinojosa, Gilberto Ochoa-Ruiz, Ketil Joachim Oedegaard, and Petter Jakobsen (petter.jakobsen@helse-bergen.no).


#############
Terms of use
#############

In all documents and papers that report experimental results based on the OBF-Psychiatric Dataset, a reference to the following paper should be included:

Enrique Garcia-Ceja, Andrea Stautland, Michael A. Riegler, Pål Halvorsen, Salvador Hinojosa, Gilberto Ochoa-Ruiz, Ketil Joachim Oedegaard, and Petter Jakobsen, "OBF-Psychiatric, a motor activity dataset of depressed, schizophrenic, and attention deficit hyperactivity disorder patients", 2024.


adhd 45 ['adhd_41.csv', 'adhd_108.csv', 'adhd_43.csv']
clinical 40 ['clinical_101.csv', 'clinical_10.csv', '

In [28]:
rows = []
for d in ['adhd','clinical','depression','control','schizophrenia']:
    for fn in os.listdir(f"{PATHS['OBF']}/{d}"):
        df = pd.read_csv(f"{PATHS['OBF']}/{d}/{fn}")
        for day, g in df.groupby('date'):
            a = g['activity'].values
            rows.append({'group': d, 'number': int(fn.split('_')[1].split('.')[0]), 'date': day,
                         'mean': a.mean(), 'sd': a.std(), 'pctZeros': (a==0).mean(), 'median': np.median(a),
                         'n_min': len(a)})
obf_clean = pd.DataFrame(rows)
print(obf_clean.groupby('group').agg(people=('number','nunique'), days=('date','size')))
obf_clean.to_csv(f'{OUT_DIR}/obf_person_day_features.csv', index=False)

               people  days
group                      
adhd               45   343
clinical           40   329
control            32   739
depression         23   405
schizophrenia      22   390


In [29]:
print(obf_clean['n_min'].describe())
print(obf_clean.groupby('group')['n_min'].apply(lambda s: (s==1440).mean()).round(2))

full = obf_clean[obf_clean['n_min']==1440]
print(full.groupby('group').agg(people=('number','nunique'), days=('date','size')))

# does the full-day subset reproduce features.csv?
u = 'control'; n = sorted(obf_clean.loc[obf_clean.group==u,'number'].unique())[0]
mine = np.sort(full.loc[(full.group==u)&(full.number==n),'mean'].values).round(3)
theirs = np.sort(feat.loc[feat['user'].astype(str)==f'{u}_{n}','mean'].values).round(3)
print(len(mine), len(theirs)); print(mine[:5]); print(theirs[:5])

count    2206.000000
mean     1339.447869
std       260.492740
min         1.000000
25%      1440.000000
50%      1440.000000
75%      1440.000000
max      1500.000000
Name: n_min, dtype: float64
group
adhd             0.74
clinical         0.76
control          0.91
depression       0.89
schizophrenia    0.87
Name: n_min, dtype: float64
               people  days
group                      
adhd               45   253
clinical           40   249
control            32   670
depression         23   359
schizophrenia      22   340
34 7
[0. 0. 0. 0. 0.]
[169.663 183.691 185.568 194.424 225.981]


In [30]:
print(feat['user'].unique()[:10])

print(obf_clean[obf_clean['n_min']>1440].groupby('group').size())
print((obf_clean['mean']==0).sum(), 'all-zero days')
print(obf_clean[obf_clean['mean']==0].groupby('group').size())

# duplicate minutes within a date
d = pd.read_csv(f"{PATHS['OBF']}/control/{sorted(os.listdir(PATHS['OBF']+'/control'))[0]}")
print(d.groupby('date').size().sort_values(ascending=False).head())
print(d[d.duplicated(['timestamp'], keep=False)].head(10))

['control_1' 'control_2' 'control_3' 'control_4' 'control_5' 'control_6'
 'control_7' 'control_8' 'control_9' 'control_10']
group
schizophrenia    6
dtype: int64
87 all-zero days
group
clinical          1
control          65
depression        5
schizophrenia    16
dtype: int64
date
2003-03-19    1440
2003-03-21    1440
2003-03-20    1440
2003-03-22    1440
2003-03-23    1440
dtype: int64
Empty DataFrame
Columns: [timestamp, date, activity]
Index: []


In [31]:
good = obf_clean[(obf_clean.n_min==1440) & (obf_clean['mean']>0)]
print(good.groupby('group').agg(people=('number','nunique'), days=('date','size')))

for n in sorted(good.loc[good.group=='control','number'].unique())[:5]:
    mine = np.sort(good.loc[(good.group=='control')&(good.number==n),'mean'].values).round(3)
    theirs = np.sort(feat.loc[feat['user']==f'control_{n}','mean'].values).round(3)
    print(n, len(mine), len(theirs), np.allclose(mine[:len(theirs)], theirs) if len(theirs) else '-')

               people  days
group                      
adhd               45   253
clinical           40   249
control            32   605
depression         23   354
schizophrenia      22   324
1 29 7 False
2 21 19 False
3 25 11 False
4 21 12 False
5 32 12 False


In [32]:
for n in [1,2,3,4,5]:
    mine = np.round(good.loc[(good.group=='control')&(good.number==n),'mean'].values, 3)
    theirs = np.round(feat.loc[feat['user']==f'control_{n}','mean'].values, 3)
    hit = np.isin(theirs, mine).mean()
    print(n, len(mine), len(theirs), f'{hit:.0%} of theirs found in mine')

1 29 7 100% of theirs found in mine
2 21 19 100% of theirs found in mine
3 25 11 100% of theirs found in mine
4 21 12 100% of theirs found in mine
5 32 12 100% of theirs found in mine


In [34]:
#
# # Section 6 — the corrected dataset and the positive control
#
# Builds the analysis the OBF data actually supports, and uses it to show the Paper 1
# diagnostics come back clean on a correctly constructed dataset.
#
# Run after cell 0 of the diagnostics notebook (needs PATHS, OUT_DIR, cv_f1, rf, macro_f1,
# perm_index, fmt). Paste as one cell, or run as a script with those names defined.
#
# 6.1  person-day features for all 162 people, from the raw series
# 6.2  join to the metadata tables on `number`
# 6.3  benchmark: diagnosis from actigraphy, subject-grouped
# 6.4  the genuinely paired subset: 85 people with same-person actigraphy AND clinical scales
# 6.5  positive control: on a real pairing, the within-class shuffle SHOULD bite

#

import os, json, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedKFold

GROUPS = ['adhd', 'clinical', 'depression', 'control', 'schizophrenia']
S6 = {}

# ---------------------------------------------------------------- 6.1 person-day features
rows = []
for g in GROUPS:
    for fn in sorted(os.listdir(f"{PATHS['OBF']}/{g}")):
        if not fn.endswith('.csv'):
            continue
        df = pd.read_csv(f"{PATHS['OBF']}/{g}/{fn}")
        number = int(fn.split('_')[1].split('.')[0])
        for day, d in df.groupby('date'):
            a = d['activity'].values.astype(float)
            rows.append({
                'group': g, 'number': number, 'person': f'{g}_{number}', 'date': day,
                'n_min': len(a), 'n_dup_ts': int(d['timestamp'].duplicated().sum()),
                'mean': a.mean(), 'sd': a.std(), 'pctZeros': float((a == 0).mean()),
                'median': float(np.median(a)), 'q25': float(np.percentile(a, 25)),
                'q75': float(np.percentile(a, 75)),
            })
pd_all = pd.DataFrame(rows)

# Filter, stated explicitly in the paper: complete days, no duplicate minutes, device worn.
keep = (pd_all.n_min == 1440) & (pd_all.n_dup_ts == 0) & (pd_all['mean'] > 0)
pdf = pd_all[keep].reset_index(drop=True)
S6['filter'] = {'person_days_raw': len(pd_all), 'person_days_kept': len(pdf),
                'dropped_incomplete': int((pd_all.n_min != 1440).sum()),
                'dropped_dup_timestamps': int((pd_all.n_dup_ts > 0).sum()),
                'dropped_zero_activity': int((pd_all['mean'] == 0).sum())}
print('person-days:', S6['filter'])
print(pdf.groupby('group').agg(people=('person', 'nunique'), days=('date', 'size')))

# ---------------------------------------------------------------- 6.2 join to metadata
info = pd.concat([pd.read_csv(f"{PATHS['OBF']}/{g}-info.csv").assign(group=g) for g in GROUPS],
                 ignore_index=True)
assert info['number'].is_unique, 'number is not globally unique; join on (group, number) instead'

# Cast both merge keys to string to prevent type-mismatch ValueError
pdf['number'] = pdf['number'].astype(str)
info['number'] = info['number'].astype(str)

D = pdf.merge(info, on=['group', 'number'], how='left', validate='many_to_one')
print(f'\njoined: {len(D)} person-days, {D.person.nunique()} people, ')

ACT = ['mean', 'sd', 'pctZeros', 'median', 'q25', 'q75']
SCALES = [c for c in ['madrs', 'hads_d', 'hads_a', 'asrs', 'mdq_pos', 'wurs'] if c in D.columns]
print('clinical scales present:', SCALES)
print('scale coverage by group:\n', D.groupby('group')[SCALES].apply(lambda g: g.notna().mean()).round(2))

# ---------------------------------------------------------------- 6.3 benchmark
#  Real diagnoses from the folder/group, not cluster labels. Grouped by person.
y_g = pd.factorize(D['group'])[0]
S6['diagnosis_5class_subject_grouped'] = cv_f1(D[ACT].values, y_g, groups=D['person'].values)
print(f"\n5-class diagnosis from actigraphy, subject-grouped: {fmt(*S6['diagnosis_5class_subject_grouped'])}")

#  Random-row CV on the same data, to size the leakage the grouped split removes.
S6['diagnosis_5class_random_rows'] = cv_f1(D[ACT].values, y_g)
print(f"  same data, random person-day split:               {fmt(*S6['diagnosis_5class_random_rows'])}")

#  Three-class version, comparable to the published baselines on features.csv.
m3 = D['group'].isin(['control', 'depression', 'schizophrenia'])
S6['diagnosis_3class_subject_grouped'] = cv_f1(D.loc[m3, ACT].values,
                                               pd.factorize(D.loc[m3, 'group'])[0],
                                               groups=D.loc[m3, 'person'].values)
print(f"  3-class (control/depression/schizophrenia):       {fmt(*S6['diagnosis_3class_subject_grouped'])}")

# ---------------------------------------------------------------- 6.4 the genuinely paired subset
#  adhd-info and clinical-info carry same-person clinical scales AND an explicit bipolar flag.
P = D[D['group'].isin(['adhd', 'clinical'])].copy()
print(f'\npaired subset: {len(P)} person-days, {P.person.nunique()} people')
for c in ['bipolar', 'unipolar']:
    if c in P.columns:
        print(f'  {c}:', P.drop_duplicates('person')[c].value_counts(dropna=False).to_dict())

#  One row per person: actigraphy averaged over that person's days, scales as recorded.
per = (P.groupby('person')
         .agg({**{c: 'mean' for c in ACT}, **{c: 'first' for c in SCALES + ['bipolar', 'group']}})
         .reset_index())
per = per.dropna(subset=SCALES + ['bipolar'])
y_bp = per['bipolar'].astype(int).values
print(f'people with complete scales + bipolar flag: {len(per)} '
      f'(bipolar={int(y_bp.sum())}, non-bipolar={int((1 - y_bp).sum())})')
S6['paired_subset'] = {'n_people': len(per), 'n_bipolar': int(y_bp.sum())}

# ---------------------------------------------------------------- 6.5 positive control
#  Same fusion design as the failed pipeline, but rows are real people.
#  Prediction: here the within-class shuffle SHOULD cost performance, because the
#  actigraphy-to-scales correspondence carries person-level information beyond the label.
if len(per) >= 30 and len(np.unique(y_bp)) == 2 and np.bincount(y_bp).min() >= 5:
    A = per[ACT].values.astype(float)
    C = per[SCALES].values.astype(float)
    A = (A - A.mean(0)) / (A.std(0) + 1e-9)
    C = (C - C.mean(0)) / (C.std(0) + 1e-9)
    X = np.hstack([A, C])
    rng = np.random.default_rng(SEED)
    reps = 10
    res = {'actigraphy_only': cv_f1(A, y_bp, n_repeats=reps)[0],
           'scales_only': cv_f1(C, y_bp, n_repeats=reps)[0],
           'fusion_aligned': cv_f1(X, y_bp, n_repeats=reps)[0]}
    wi = [cv_f1(np.hstack([A, C[perm_index(len(y_bp), rng, y_bp)]]), y_bp, n_repeats=1, seed=SEED + i)[0]
          for i in range(reps)]
    ac = [cv_f1(np.hstack([A, C[perm_index(len(y_bp), rng)]]), y_bp, n_repeats=1, seed=SEED + i)[0]
          for i in range(reps)]
    res['fusion_within_class_shuffle'] = (float(np.mean(wi)), float(np.std(wi)))
    res['fusion_across_class_shuffle'] = (float(np.mean(ac)), float(np.std(ac)))
    S6['positive_control'] = res
    print('\nPositive control — genuine same-person pairing (bipolar flag, n=%d)' % len(per))
    for k in ['actigraphy_only', 'scales_only', 'fusion_aligned']:
        print(f'  {k:30s} {res[k]:.3f}')
    print(f"  {'fusion_within_class_shuffle':30s} {fmt(*res['fusion_within_class_shuffle'])}")
    print(f"  {'fusion_across_class_shuffle':30s} {fmt(*res['fusion_across_class_shuffle'])}")
    gap = res['fusion_aligned'] - res['fusion_within_class_shuffle'][0]
    print(f'\n  within-class gap: {gap:+.3f}   '
          f'("diagnostic fires as intended" if gap > 0.02 else "no cross-modal signal detected either")')
else:
    print('\nPositive control skipped: too few people or classes after dropping missing scales.')
    S6['positive_control'] = None

#  Missingness probe on the corrected data: should be at chance, unlike the 0.542 in the broken build.
miss = D[ACT + SCALES].isna().values.astype(int)
if miss.std(0).sum() > 0:
    S6['missingness_probe'] = cv_f1(miss, y_g, groups=D['person'].values, n_repeats=1)
    print(f"\nMissingness probe on corrected data (5-class, floor 0.200): "
          f"{fmt(*S6['missingness_probe'])}")
    print('  NB: scales are structurally absent for 3 of 5 groups, so this is expected to fire.')
    print('  Report it for the actigraphy-only matrix too:')
    m2 = D[ACT].isna().values.astype(int)
    S6['missingness_probe_actigraphy_only'] = (float(m2.std()), 'all-present' if m2.sum() == 0 else 'varies')
    print('   ', S6['missingness_probe_actigraphy_only'])

# ---------------------------------------------------------------- save
D.to_csv(f'{OUT_DIR}/obf_corrected_person_day.csv', index=False)
per.to_csv(f'{OUT_DIR}/obf_paired_person_level.csv', index=False)
with open(f'{OUT_DIR}/section6.json', 'w') as f:
    json.dump(S6, f, indent=2, default=lambda o: float(o) if isinstance(o, np.floating) else str(o))
print('\n=== PASTE BACK ===')
print(json.dumps(S6, default=lambda o: round(float(o), 3) if isinstance(o, (float, np.floating)) else str(o)))
print('=== END ===')

person-days: {'person_days_raw': 2206, 'person_days_kept': 1785, 'dropped_incomplete': 335, 'dropped_dup_timestamps': 6, 'dropped_zero_activity': 87}
               people  days
group                      
adhd               45   253
clinical           40   249
control            32   605
depression         23   354
schizophrenia      22   324

joined: 1785 person-days, 162 people, 
clinical scales present: ['madrs', 'hads_d', 'hads_a', 'asrs', 'mdq_pos', 'wurs']
scale coverage by group:
                madrs  hads_d  hads_a  asrs  mdq_pos  wurs
group                                                    
adhd             0.0     0.0     0.0   0.0      0.0   0.0
clinical         0.0     0.0     0.0   0.0      0.0   0.0
control          0.0     0.0     0.0   0.0      0.0   0.0
depression       0.0     0.0     0.0   0.0      0.0   0.0
schizophrenia    0.0     0.0     0.0   0.0      0.0   0.0

5-class diagnosis from actigraphy, subject-grouped: 0.314 ± 0.008
  same data, random person-day sp

In [35]:
print(info['number'].head().tolist(), info['number'].dtype)
print(pdf['person'].head().tolist())

['adhd_1', 'adhd_2', 'adhd_3', 'adhd_5', 'adhd_8'] object
['adhd_1', 'adhd_1', 'adhd_1', 'adhd_1', 'adhd_1']


In [39]:
info = pd.concat([pd.read_csv(f"{PATHS['OBF']}/{g}-info.csv").assign(group=g) for g in GROUPS], ignore_index=True)
info['person'] = info['group'] + '_' + info['number'].astype(str)

# Ensure identical types for both merge keys
pdf['number'] = pdf['number'].astype(str)
info['number'] = info['number'].astype(str)

# Merge on both matching keys to avoid duplicate columns and suffix generation
D = pdf.merge(info.drop(columns=['group']), on=['person', 'number'], how='left', validate='many_to_one')
print('unmatched person-days:', D['number'].isna().sum(), '| people matched:', D.loc[D['number'].notna(), 'person'].nunique(), 'of', pdf['person'].nunique())

ACT = ['mean','sd','pctZeros','median','q25','q75']
SCALES = [c for c in ['madrs','hads_d','hads_a','asrs','mdq_pos','wurs'] if c in D.columns]
print(D.groupby('group')[SCALES].apply(lambda g: g.notna().mean()).round(2))

unmatched person-days: 0 | people matched: 162 of 162
               madrs  hads_d  hads_a  asrs  mdq_pos  wurs
group                                                    
adhd             0.0     0.0     0.0   0.0      0.0   0.0
clinical         0.0     0.0     0.0   0.0      0.0   0.0
control          0.0     0.0     0.0   0.0      0.0   0.0
depression       0.0     0.0     0.0   0.0      0.0   0.0
schizophrenia    0.0     0.0     0.0   0.0      0.0   0.0


In [40]:
info = pd.concat([pd.read_csv(f"{PATHS['OBF']}/{g}-info.csv").assign(group=g) for g in GROUPS],
                 ignore_index=True)
info['person'] = info['number'].astype(str)      # already 'adhd_1' — no prefix to add

# pdf['number'] is the int from the filename; info['number'] is the full id. Drop ours.
D = (pdf.drop(columns=['number'])
        .merge(info.drop(columns=['group', 'number']), on='person', how='left',
               validate='many_to_one'))

# check a right-hand-only column
print('unmatched person-days:', D['age'].isna().sum(),
      '| people with metadata:', D.loc[D['age'].notna(), 'person'].nunique(), 'of', pdf['person'].nunique())

ACT = ['mean','sd','pctZeros','median','q25','q75']
SCALES = [c for c in ['madrs','hads_d','hads_a','asrs','mdq_pos','wurs'] if c in D.columns]
print(D.groupby('group')[SCALES].apply(lambda g: g.notna().mean()).round(2))
print(D.drop_duplicates('person').groupby('group')[['bipolar','unipolar']].apply(lambda g: g.notna().mean()).round(2))

unmatched person-days: 0 | people with metadata: 162 of 162
               madrs  hads_d  hads_a  asrs  mdq_pos  wurs
group                                                    
adhd            0.94    0.92    0.92  0.98     0.95  0.98
clinical        0.94    0.92    0.92  0.92     0.90  0.92
control         0.00    0.00    0.00  0.00     0.00  0.00
depression      0.00    0.00    0.00  0.00     0.00  0.00
schizophrenia   0.00    0.00    0.00  0.00     0.00  0.00
               bipolar  unipolar
group                           
adhd               1.0       1.0
clinical           1.0       1.0
control            0.0       0.0
depression         0.0       0.0
schizophrenia      0.0       0.0


In [41]:
# 6.3
y_g = pd.factorize(D['group'])[0]
S6['diagnosis_5class_subject_grouped'] = cv_f1(D[ACT].values, y_g, groups=D['person'].values)
S6['diagnosis_5class_random_rows']     = cv_f1(D[ACT].values, y_g)
m3 = D['group'].isin(['control','depression','schizophrenia'])
S6['diagnosis_3class_subject_grouped'] = cv_f1(D.loc[m3, ACT].values,
                                               pd.factorize(D.loc[m3,'group'])[0],
                                               groups=D.loc[m3,'person'].values)
for k in ['diagnosis_5class_subject_grouped','diagnosis_5class_random_rows','diagnosis_3class_subject_grouped']:
    print(f'{k:36s} {fmt(*S6[k])}')

# 6.4
P = D[D['group'].isin(['adhd','clinical'])]
print(P.drop_duplicates('person')['bipolar'].value_counts(dropna=False).to_dict())

per = (P.groupby('person')
         .agg({**{c:'mean' for c in ACT}, **{c:'first' for c in SCALES+['bipolar','group']}})
         .reset_index()
         .dropna(subset=SCALES+['bipolar']))
y_bp = per['bipolar'].astype(int).values
print(f'{len(per)} people | bipolar={int(y_bp.sum())} non-bipolar={int((1-y_bp).sum())}')

diagnosis_5class_subject_grouped     0.314 ± 0.008
diagnosis_5class_random_rows         0.400 ± 0.004
diagnosis_3class_subject_grouped     0.523 ± 0.016
{0.0: 49, 1.0: 36}
71 people | bipolar=30 non-bipolar=41


In [42]:
A = per[ACT].values.astype(float)
C = per[SCALES].values.astype(float)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
C = (C - C.mean(0)) / (C.std(0) + 1e-9)
X = np.hstack([A, C])

rng = np.random.default_rng(SEED); reps = 10
res = {'actigraphy_only': cv_f1(A, y_bp, n_repeats=reps)[0],
       'scales_only':     cv_f1(C, y_bp, n_repeats=reps)[0],
       'fusion_aligned':  cv_f1(X, y_bp, n_repeats=reps)[0]}
wi = [cv_f1(np.hstack([A, C[perm_index(len(y_bp), rng, y_bp)]]), y_bp, n_repeats=1, seed=SEED+i)[0] for i in range(reps)]
ac = [cv_f1(np.hstack([A, C[perm_index(len(y_bp), rng)]]),       y_bp, n_repeats=1, seed=SEED+i)[0] for i in range(reps)]
res['fusion_within_class_shuffle'] = (float(np.mean(wi)), float(np.std(wi)))
res['fusion_across_class_shuffle'] = (float(np.mean(ac)), float(np.std(ac)))
S6['positive_control'] = res
for k,v in res.items(): print(f'{k:30s}', fmt(*v) if isinstance(v, tuple) else f'{v:.3f}')
print(f"within-class gap: {res['fusion_aligned'] - res['fusion_within_class_shuffle'][0]:+.3f}")

actigraphy_only                0.494
scales_only                    0.733
fusion_aligned                 0.732
fusion_within_class_shuffle    0.726 ± 0.027
fusion_across_class_shuffle    0.486 ± 0.072
within-class gap: +0.006


In [43]:
chest = [c for c in WESAD_FEATURES if c.startswith('chest')]
wrist = [c for c in WESAD_FEATURES if c.startswith('wrist')]
w = wesad_all[wesad_all.label.isin([1,2,3,4])]
rng_ = np.random.default_rng(SEED)
s = np.concatenate([rng_.choice(np.where(w.label.values==c)[0], 1500, replace=False) for c in [1,2,3,4]])
Aw, Cw = w.iloc[s][chest].values, w.iloc[s][wrist].values
yw, gw = pd.factorize(w.iloc[s].label)[0], w.iloc[s].subject.values
print('aligned ', cv_f1(np.hstack([Aw,Cw]), yw, groups=gw, n_repeats=3))
print('within  ', cv_f1(np.hstack([Aw, Cw[perm_index(len(yw), rng_, yw)]]), yw, groups=gw, n_repeats=3))

aligned  (0.42202760852669235, 0.018835497049005737)
within   (0.6015546412944102, 0.02353851123965798)


In [44]:
def perm_within_cells(y, g, rng):
    idx = np.arange(len(y))
    for key in set(zip(g, y)):
        pos = np.where((g == key[0]) & (y == key[1]))[0]
        idx[pos] = rng.permutation(pos)
    return idx

rng_ = np.random.default_rng(SEED)
pw = perm_within_cells(yw, gw, rng_)
print('aligned        ', cv_f1(np.hstack([Aw, Cw]),     yw, groups=gw, n_repeats=3))
print('within-cell    ', cv_f1(np.hstack([Aw, Cw[pw]]), yw, groups=gw, n_repeats=3))

aligned         (0.42202760852669235, 0.018835497049005737)
within-cell     (0.4329793064056191, 0.020548754754641525)
